In [ ]:
import os
import io
import sys

import logging
import logging.handlers

from datetime import date
import time

from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional
import warnings

import numpy as np
import pandas as pd
import scipy.stats as st

import json
import pickle

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_validate, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.float_format", "{:.4f}".format)

data_path = os.path.join("..", "..", "data")
processed_data_path = os.path.join(data_path, 'processed_data')
ann_folder_path = os.path.join(processed_data_path, 'ann_folder_path')
visualisation_path = os.path.join("..", "..", "0_documentation", "visualisations")

## Logging

### Initialise Log Info

In [ ]:
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

# will be used to date the trained model
today = str(date.today())
today = today.replace("-","")

log_name = f'{today}_xgboost.log'

### Recreate or Reintialise Log

In [ ]:
recreate_log = True

if recreate_log and os.path.exists(os.path.join(log_dir, log_name)):
    os.remove(os.path.join(log_dir, log_name))

In [ ]:
log = logging.getLogger(log_name)

log.setLevel(logging.DEBUG)
    
fmt_plain = logging.Formatter("%(asctime)s [%(levelname)s] %(name)s — %(message)s",
                              datefmt="%Y-%m-%d %H:%M:%S")

if not log.handlers:
    # Rotating plain-text file
    fh = logging.handlers.RotatingFileHandler(log_dir / f"{log_name}")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt_plain)
    log.addHandler(fh)

    log.info(f"log initialised — outputs: %s/{log_name}", log_name)
else:
    log.info(r"ML Process Restarted")
    log.info(f"log initialised — outputs: %s/{log_name}", log_name)

#### Create Metric Logging Function

In [ ]:
def log_metric(key: str, value, step: Optional[str] = None) -> None:
    """Log a named metric — appears in console output and JSON Lines."""
    extra = {"metric": key, "value": value}
    if step:
        extra["step"] = step
    log.info("METRIC  %-35s = %s", key, value, extra={"extra": extra})

## Import Data

In [ ]:
log.info("Loading Data for ML")

df = pd.read_csv(os.path.join(data_path, "20260313_ckd_data_for_ml.csv"))

log.info(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} features | class balance {df['Endpoint'].sum()/df.shape[0]:.2%}")

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} features | class balance {df['Endpoint'].sum()/df.shape[0]:.2%}")

## Structure Data for Modelling

### Remove Sparse Columns

In [ ]:
log.info(f"Prepare DataFrame for ML")
log.info(f"Dropping Sparse Features")

drop_cols = ['MasterPersonID', 'IMDRank', 'firstGFRClassification', 'lastGFRClassification', 'firstBMICategorisation',
             'lastBMICategorisation', 'firstBPCategorisation', 'lastBPCategorisation', 'EndpointType']

rows = df.shape[0]

for col in df.columns:
    if col not in drop_cols:
        col_count = df[col].count()
        count_percent = col_count/rows
    
        col_sum = df[col].sum()
        sum_percent = count_percent if type(col_sum) == str or col_sum/rows < 0 else col_sum/rows
    
        if count_percent < 0.5 or sum_percent < 0.01:
            drop_cols.append(col)

drop_cols.remove('SmokingStatus_OccasionalSmoker')

log.info(f"Columns Dropped due to being sparsly populated: {drop_cols}")

df = df[df['IMDDecile'].notna()].drop(columns=drop_cols).reset_index(drop=True)

del drop_cols, rows, col_count, count_percent, col_sum, sum_percent

df['IMDDecile'] = df['IMDDecile'].astype(int)

### Fill Missing Data with Median Feature Values

In [ ]:
log.info(f"Fill Missing Blood Results with Mean Values")

for col in df.columns:
    if df[col].dtype == 'float64':
        df[col] = df[col].fillna(df[col].mean())

df.head()

In [ ]:
df['Endpoint'].value_counts(normalize=True)

## Run Modelling

### Train Test Split

In [ ]:
log.info(f"Create X & y Variables")
X = df.drop(columns=['Endpoint'])
y = df['Endpoint']

In [ ]:
test_size = 0.3
random_state = 144
log.info(f"Create Train & Test Split with a Test Size of {test_size:.0%}")

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=test_size,
                                                    stratify=y,
                                                    random_state=random_state,
                                                   )

log.info(f"Train/test split: {len(X_train):,} / {len(X_test):,}")

print(f"Train/test split: {len(X_train):,} / {len(X_test):,}")

### Pipeline Creation

In [ ]:
C = 1.0
solver = "lbfgs"
penalty = "l2"
max_iter = 1_000

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", XGBClassifier(eval_metric="logloss", random_state=random_state))
    ])

log.info(f"XGBoost Classifier Pipeline including Standard Scaling created.")

### Create Search Strategy

In [ ]:
search_strategy = "grid" # "grid" | "random"
cv_folds = 25
n_iter = 20 # only used for random search

param_grid = {
    "clf__n_estimators":  [100, 300, 500],
    "clf__max_depth":     [3, 5, 7],
    "clf__learning_rate": [0.01, 0.1, 0.3],
    "clf__subsample":     [0.7, 1.0],
    "clf__colsample_bytree": [0.7, 1.0]
    }

### Run Search Strategy

In [ ]:
cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)

In [ ]:
log.info(f"Starting {search_strategy} hyperparameter search (cv={cv_folds})")

cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
t0 = time.perf_counter()

if search_strategy == "grid":
    searcher = GridSearchCV(pipeline, param_grid, cv=cv, scoring="roc_auc", n_jobs=1, verbose=1)
else:
    searcher = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=n_iter, cv=cv, scoring="roc_auc", n_jobs=1, verbose=1, random_state=random_state)

try:
    searcher.fit(X_train, y_train)
except ValueError as e:
    log.error(f'Value Error: {e}')
    raise Exception(f'Value Error: {e}')
except NameError as e:
    log.error(f'Name Error: {e}')
    raise Exception(f'Name Error: {e}')
except IndexError as e:
    log.error(f'Index Error: {e}')
    raise Exception(f'Index Error: {e}')
except LookupError as e:
    log.error(f'Lookup Error: {e}')
    raise Exception(f'Lookup Error: {e}')
except TypeError as e:
    log.error(f'Type Error: {e}')
    raise Exception(f'Type Error: {e}')
except RuntimeError as e:
    log.error(f'Runtime Error: {e}')
    raise (f'Runtime Error: {e}')
except ReferenceError as e:
    log.error(f'Reference Error: {e}')
    raise Exception(f'Reference Error: {e}')
except AttributeError as e:
    log.error(f'Attribute Error: {e}')
    raise Exception(f'Attribute Error: {e}')

search_duration = time.perf_counter() - t0

In [ ]:
try:
    best_pipeline = searcher.best_estimator_
except ValueError as e:
    log.error(f'Value Error: {e}')
    raise Exception(f'Value Error: {e}')
except NameError as e:
    log.error(f'Name Error: {e}')
    raise Exception(f'Name Error: {e}')
except IndexError as e:
    log.error(f'Index Error: {e}')
    raise Exception(f'Index Error: {e}')
except LookupError as e:
    log.error(f'Lookup Error: {e}')
    raise Exception(f'Lookup Error: {e}')
except TypeError as e:
    log.error(f'Type Error: {e}')
    raise Exception(f'Type Error: {e}')
except RuntimeError as e:
    log.error(f'Runtime Error: {e}')
    raise (f'Runtime Error: {e}')
except ReferenceError as e:
    log.error(f'Reference Error: {e}')
    raise Exception(f'Reference Error: {e}')
except AttributeError as e:
    log.error(f'Attribute Error: {e}')
    raise Exception(f'Attribute Error: {e}')

log.info(f"Best params : {searcher.best_params_}")
log_metric("hyperparam_best_roc_auc",    round(searcher.best_score_, 4), step="hyperparam")
log_metric("hyperparam_search_duration", round(search_duration, 2),      step="hyperparam")

In [ ]:
with open(os.path.join(processed_data_path, f"{today}_ckd_xgboost_model.pickle"), "wb") as f:
    pickle.dump(best_pipeline, f)

log.info(f"Best Model Saved to {processed_data_path}/{today}_ckd_xgboost_model.pickle")

#### Visualise Parameter Search

In [ ]:
cv_res = pd.DataFrame(searcher.cv_results_)
top20  = cv_res.sort_values("mean_test_score", ascending=False).head(20).reset_index(drop=True)

labels = [
    f"Learning Rate={r['param_clf__learning_rate']}  {r['param_clf__n_estimators']}"
    for _, r in top20.iterrows()
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(top20)), top20["mean_test_score"],
        xerr=top20["std_test_score"],
        align="center", color="steelblue", alpha=0.8, capsize=3)
ax.set_yticks(range(len(top20)), labels, fontsize=8)
ax.set_xlabel("Mean CV ROC-AUC")
ax.set_title("Hyperparameter Search — Top 20 Combinations", fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()

plt.savefig(os.path.join(visualisation_path, f'{today}_xgboost_paramater_search.png'))
log.info(f'Paramater Search Plot Saved to {visualisation_path}/{today}_xgboost_parameter_search.png')

plt.show()

### Cross Validation

In [ ]:
log.info(f"Running {cv_folds}-fold stratified cross-validation")

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "neg_mean_squared_error" : "neg_mean_squared_error"
}

In [ ]:
try:
    if best_pipeline:
        print('')
except:
    today = '20260430'
    with open(os.path.join(processed_data_path, f"{today}_ckd_xgboost_model.pickle"), "rb") as f:
        best_pipeline = pickle.load(f)

t0 = time.perf_counter()

try:
    cv_out = cross_validate(best_pipeline, X_train, y_train,
                            cv=StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state),
                            scoring=scoring, return_train_score=True, n_jobs=-1, verbose=1)
except ValueError as e:
    log.error(f'Value Error: {e}')
    raise Exception(f'Value Error: {e}')
except NameError as e:
    log.error(f'Name Error: {e}')
    raise Exception(f'Name Error: {e}')
except IndexError as e:
    log.error(f'Index Error: {e}')
    raise Exception(f'Index Error: {e}')
except LookupError as e:
    log.error(f'Lookup Error: {e}')
    raise Exception(f'Lookup Error: {e}')
except TypeError as e:
    log.error(f'Type Error: {e}')
    raise Exception(f'Type Error: {e}')
except RuntimeError as e:
    log.error(f'Runtime Error: {e}')
    raise (f'Runtime Error: {e}')
except ReferenceError as e:
    log.error(f'Reference Error: {e}')
    raise Exception(f'Reference Error: {e}')
except AttributeError as e:
    log.error(f'Attribute Error: {e}')
    raise Exception(f'Attribute Error: {e}')

cv_duration = time.perf_counter() - t0

In [ ]:
# Build a display table with a mean row
fold_df = pd.DataFrame({
    "Fold":      range(1, cv_folds + 1),
    "Accuracy":  cv_out["test_accuracy"].round(4),
    "Precision": cv_out["test_precision"].round(4),
    "Recall":    cv_out["test_recall"].round(4),
    "F1":        cv_out["test_f1"].round(4),
    "ROC-AUC":   cv_out["test_roc_auc"].round(4),
    "Train AUC": cv_out["train_roc_auc"].round(4),
    })

mean_row = fold_df.drop(columns="Fold").mean().round(4).to_dict()
mean_row["Fold"] = "Mean"
fold_df = pd.concat([fold_df, pd.DataFrame([mean_row])], ignore_index=True)

for metric, col in [("cv_mean_accuracy", "Accuracy"), ("cv_mean_f1", "F1"), ("cv_mean_roc_auc", "ROC-AUC")]:
    log_metric(metric, float(fold_df.iloc[:-1][col].astype(float).mean().round(4)), step="cross_validation")

log.info(f"CV done in {cv_duration:.1f}s")

fold_df.style \
    .highlight_max(subset=["Accuracy", "F1", "ROC-AUC"], color="#d4edda") \
    .highlight_min(subset=["Accuracy", "F1", "ROC-AUC"], color="#f8d7da") \
    .format(precision=4)

#### Visualise Cross Validation Results

In [ ]:
plot_df = fold_df[fold_df["Fold"] != "Mean"].copy()
plot_df["Fold"] = plot_df["Fold"].astype(int)
plot_metrics = ["Accuracy", "F1", "ROC-AUC"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
for ax, metric in zip(axes, plot_metrics):
    vals = plot_df[metric].astype(float).values
    ax.bar(plot_df["Fold"], vals, color="steelblue", alpha=0.8, width=0.6)
    ax.axhline(vals.mean(), color="crimson", linestyle="--", linewidth=1.4,
               label=f"Mean = {vals.mean():.4f}")
    ax.set_title(metric, fontweight="bold")
    ax.set_xlabel("Fold")
    ax.set_xticks(plot_df["Fold"])
    ax.legend(fontsize=8)
    ax.set_ylim(max(0, vals.min() - 0.05), min(1.0, vals.max() + 0.05))

fig.suptitle(f"{cv_folds}-Fold Cross-Validation Metrics", fontsize=13, fontweight="bold")
plt.tight_layout()

plt.savefig(os.path.join(visualisation_path, f'{today}_xgboost_cross_validation.png'))
log.info(f'Cross Validation Search Plot Saved to {visualisation_path}/{today}_xgboost_cross_validation.png')

plt.show()

### Retrain Best Pipeline

In [ ]:
log.info("Retraining best pipeline on full training set")
best_pipeline.fit(X_train, y_train)

print("Model retrained on full training set")
try:
    print(f"Params: {searcher.best_params_}")
except:
    pass

### Feature Importance

In [ ]:
log.info("Analysing feature importance")

clf      = best_pipeline.named_steps["clf"]
features = X_test.columns.tolist()
importances = clf.feature_importances_

coef_df = pd.DataFrame({
    "feature":     features,
    "importance":  importances,
    "abs_coef":    importances,
}).sort_values("importance", ascending=False)

perm = permutation_importance(
    best_pipeline, X_test, y_test,
    n_repeats=15, random_state=42, n_jobs=1, scoring="roc_auc",
)

perm_df = pd.DataFrame({
    "feature":              features,
    "perm_importance_mean": perm.importances_mean,
    "perm_importance_std":  perm.importances_std,
}).sort_values("perm_importance_mean", ascending=False)

importance_df = coef_df.merge(perm_df, on="feature")
log.info("Feature importance computed")

importance_df.head(10).style.background_gradient(cmap="Blues", subset=["abs_coef", "perm_importance_mean"]).format(precision=4)

#### Visualise Feature Importnace

In [ ]:
TOP_N = 25
top_coef = importance_df.head(TOP_N).copy()
top_perm = importance_df.sort_values("perm_importance_mean", ascending=False).head(TOP_N)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Coefficients (coloured by sign)
colors = ["#e74c3c" if c > 0 else "#3498db" for c in top_coef["importance"]]
ax1.barh(top_coef["feature"], top_coef["importance"], color=colors, alpha=0.85)
ax1.axvline(0, color="black", linewidth=0.8)
ax1.set_title("Feature Importances (Gini / Gain)", fontweight="bold")
ax1.set_xlabel("Importance score")
ax1.invert_yaxis()

# Permutation importance
ax2.barh(top_perm["feature"], top_perm["perm_importance_mean"],
         xerr=top_perm["perm_importance_std"],
         color="steelblue", alpha=0.85, capsize=3)
ax2.set_title("Permutation Importance (ROC-AUC drop)", fontweight="bold")
ax2.set_xlabel("Mean importance ± std")
ax2.invert_yaxis()

fig.suptitle(f"Feature Importance — Top {TOP_N}", fontsize=13, fontweight="bold")
plt.tight_layout()

plt.savefig(os.path.join(visualisation_path, f'{today}_xgboost_feature_importance.png'))
log.info(f'Feature Importance Plot Saved to {visualisation_path}/{today}_xgboost_feature_importance.png')

plt.show()

### Benchmarking

In [ ]:
log.info(f"Benchmarking on held-out test set ({len(X_test)} samples)")

t0      = time.perf_counter()
y_pred  = best_pipeline.predict(X_test)
y_prob  = best_pipeline.predict_proba(X_test)[:, 1]
elapsed = (time.perf_counter() - t0) * 1000


t0      = time.perf_counter()
y_train_pred  = best_pipeline.predict(X_train)
y_train_prob  = best_pipeline.predict_proba(X_train)[:, 1]
elapsed_train = (time.perf_counter() - t0) * 1000

metrics = {
    "Accuracy (Test)":   round(accuracy_score(y_test, y_pred), 4),
    "Precision (Test)":  round(precision_score(y_test, y_pred, zero_division=0), 4),
    "Recall (Test)":     round(recall_score(y_test, y_pred, zero_division=0), 4),
    "F1 (Test)":         round(f1_score(y_test, y_pred, zero_division=0), 4),
    "ROC-AUC (Test)":    round(roc_auc_score(y_test, y_prob), 4),
    "ms/sample (Test)":  round(elapsed / len(X_test), 4),
    "Accuracy (Train)":  round(accuracy_score(y_train, y_train_pred), 4),
    "Precision (Train)": round(precision_score(y_train, y_train_pred, zero_division=0), 4),
    "Recall (Train)":    round(recall_score(y_train, y_train_pred, zero_division=0), 4),
    "F1 (Train)":        round(f1_score(y_train, y_train_pred, zero_division=0), 4),
    "ROC-AUC (Train)":   round(roc_auc_score(y_train, y_train_prob), 4),
    "ms/sample (Train)": round(elapsed / len(X_train), 4)
}

for k, v in metrics.items():
    log_metric(f"test_{k.lower().replace('-','_').replace('/','_')}", v, step="benchmark")

log.info("Benchmark complete")
pd.DataFrame(metrics.items(), columns=["Metric", "Value"]).style.format({"Value": "{:.4f}"})

#### Visualise Benchmarking

##### Confusion Matrix & ROC Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["Class 0", "Class 1"]).plot(ax=ax1, colorbar=False, cmap="Blues")
ax1.set_title("Confusion Matrix", fontweight="bold")

RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax2, color="steelblue")
ax2.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random")
ax2.set_title("ROC Curve", fontweight="bold")
ax2.legend()

fig.suptitle("Model Evaluation — Test Set", fontsize=13, fontweight="bold")
plt.tight_layout()

plt.savefig(os.path.join(visualisation_path, f'{today}_xgboost_confusion_matrix_and_roc_curve.png'))
log.info(f'First Benchmarking Plot Saved to {visualisation_path}/{today}_xgboost_confusion_matrix_and_roc_curve.png')

plt.show()

##### Predicted Probability Distribution by True Class

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for cls, label, color in [(0, "Class 0", "#3498db"), (1, "Class 1", "#e74c3c")]:
    ax.hist(y_prob[y_test == cls], bins=40, alpha=0.6,
            label=label, color=color, edgecolor="white")

ax.axvline(0.5, color="black", linestyle="--", linewidth=1, label="Threshold = 0.5")
ax.set_xlabel("Predicted probability (Class 1)")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution", fontweight="bold")
ax.legend()

plt.tight_layout()

plt.savefig(os.path.join(visualisation_path, f'{today}_xgboost_predicted_probs_by_true_class.png'))
log.info(f'Second Benchmarking Plot Saved to {visualisation_path}/{today}_xgboost_predicted_probs_by_true_class.png')

plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Class 0", "Class 1"]))

### Prediction Validation

In [ ]:
log.info("Running prediction validation checks")

checks = {
    "No NaN in predictions":   not np.isnan(y_pred).any(),
    "No NaN in probabilities": not np.isnan(y_prob).any(),
    "Probabilities in [0, 1]": bool((y_prob >= 0).all() and (y_prob <= 1).all()),
    "Predictions are binary":  set(np.unique(y_pred)).issubset({0, 1}),
    "Both classes predicted":  len(np.unique(y_pred)) > 1,
}

rows = []
all_passed = True
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    emoji  = "OK" if passed else "!!"
    rows.append({"Check": check, "Result": f"[{emoji}] {status}"})
    (log.info if passed else log.warning)("[%s] %s", status, check)
    if not passed:
        all_passed = False

checks_df = pd.DataFrame(rows)
display(checks_df.style.apply(
    lambda col: ["background-color: #d4edda" if "PASS" in v
                 else "background-color: #f8d7da" for v in col],
    subset=["Result"]
    ))

if all_passed:
    log.info("All prediction validation checks passed")
    print("\nAll checks passed.")
else:
    log.warning("One or more validation checks FAILED")
    print("\nOne or more checks FAILED — review output carefully.")

## Summary Dashboard

In [ ]:
top_perm_feat = importance_df.sort_values("perm_importance_mean", ascending=False)

print("=" * 58)
print("              PIPELINE SUMMARY")
print("=" * 58)
print(f"  Search strategy   : {search_strategy}")
#print(f"  Best params       : {searcher.best_params_}")
#print(f"  Search duration   : {search_duration:.1f}s")
print()
print(f"  Cross-validation ({cv_folds}-fold on training data):")
print(f"    Accuracy  : {cv_out['test_accuracy'].mean():.4f}  ±  {cv_out['test_accuracy'].std():.4f}")
print(f"    F1        : {cv_out['test_f1'].mean():.4f}  ±  {cv_out['test_f1'].std():.4f}")
print(f"    ROC-AUC   : {cv_out['test_roc_auc'].mean():.4f}  ±  {cv_out['test_roc_auc'].std():.4f}")
print()
print("  Hold-out test set:")
for k, v in metrics.items():
    print(f"    {k:<12}: {v}")
print()
print(f"  Top feature (coef): {importance_df.iloc[0]['feature']}  "
      f"({importance_df.iloc[0]['importance']:+.4f})")
print(f"  Top feature (perm): {top_perm_feat.iloc[0]['feature']}  "
      f"({top_perm_feat.iloc[0]['perm_importance_mean']:.4f})")
print()
all_ok = all(checks.values())
print(f"  Validation checks : {'ALL PASSED' if all_ok else 'ONE OR MORE FAILED'}")
print(f"  Artefacts saved   : {log_dir}/")
print("=" * 58)

## Save Results

In [ ]:
all_results = {
    "hyperparameter_search": {
        "strategy":        search_strategy,
        "best_params":     searcher.best_params_,
        "best_cv_roc_auc": round(searcher.best_score_, 4),
        "duration_s":      round(search_duration, 2),
    },
    "cross_validation": {
        "mean_accuracy":  float(cv_out["test_accuracy"].mean().round(4)),
        "std_accuracy":   float(cv_out["test_accuracy"].std().round(4)),
        "mean_f1":        float(cv_out["test_f1"].mean().round(4)),
        "std_f1":         float(cv_out["test_f1"].std().round(4)),
        "mean_roc_auc":   float(cv_out["test_roc_auc"].mean().round(4)),
        "std_roc_auc":    float(cv_out["test_roc_auc"].std().round(4)),
    },
    "feature_importance": importance_df.head(10).to_dict(orient="records"),
    "benchmark":          metrics,
    "validation_checks":  {k: bool(v) for k, v in checks.items()},
}

out_path = os.path.join(ann_folder_path, f"{today}_xgboost__run_results.json")
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2, default=str)

log.info(f"Full results saved to {out_path}")
print(f"Results saved to {out_path}")
print(f"Log files in   : {log_dir}/")

## End

## Sandbox

In [ ]:
pd.DataFrame().from_dict(cv_out)

In [ ]:
training_score = best_pipeline.score(X_train, y_train)

training_score

In [ ]:
train_size, train_scores, test_scores = learning_curve(best_pipeline, X=X, y=y, cv=cv, scoring="accuracy", random_state=random_state)

train_scores = 1-np.mean(train_scores,axis=1)#converting the accuracy score to misclassification rate
test_scores = 1-np.mean(test_scores,axis=1)#converting the accuracy score to misclassification rate

In [ ]:
lc = pd.DataFrame({"Training_size": train_size,
                   "Training_loss": train_scores,
                   "Validation_loss": test_scores}).melt(id_vars="Training_size")

In [ ]:
lc_detals = {"cv_scores": cv_out,
             "train_score": training_score,
             "learning_curve": lc}

In [ ]:
sns.lineplot(data=lc_detals["learning_curve"], x="Training_size", y="value", hue="variable")
plt.title("Learning Curve of Good Fit Model")
plt.ylabel("Misclassification Rate/Loss");

In [ ]:
scores_df = pd.DataFrame().from_dict(cv_out)[['test_neg_mean_squared_error', 'train_neg_mean_squared_error']]

plt.figure(figsize=(10,4))

# Changing colours, order of the bars, etc
g = sns.barplot(
    data=scores_df,
    errorbar='sd',
    palette='pastel',
    capsize=0.1,
)

# Changing font sizes and adding labels
g.set_title('Model performance', fontsize=22)
g.set_xlabel('Metric')
g.set_ylabel('Score')
#g.set_ylim(0.8, 1.05)

# Rotating labels - useful if they are too long and overlap.
plt.xticks(rotation=-90)

plt.show()

## File End